In [1]:
import cv2
import time
import csv
from ultralytics import YOLO

# Configuration
INPUT_VIDEO = r"D:\Documents\University\Adv Proj\Vdo-Demo-Tester\demo1(dawn).mp4"
OUTPUT_VIDEO = r"D:\Documents\University\Adv Proj\output_demo\Demo3\YOLO\processed_yolo.mp4"
CSV_LOG_PATH = r"D:\Documents\University\Adv Proj\output_demo\Demo3\YOLO\yolo_detection_log.csv"
MODEL_PATH = r"D:\Documents\University\Adv Proj\Train Model\YOLO\V-1.0\yolo_driver_v1\weights\best.pt"

print("Loading YOLOv8 model")
model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(INPUT_VIDEO)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

with open(CSV_LOG_PATH, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Frame_ID", "Detection_Class", "Confidence_Score", "Inference_Time_ms"])

    frame_id = 0
    print("Processing video")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        start = time.perf_counter()
        results = model(frame, verbose=False)
        end = time.perf_counter()
        
        latency = (end - start) * 1000

        best_class = "None"
        best_conf = 0.0

        if len(results[0].boxes) > 0:
            top_box = results[0].boxes[0]
            best_conf = float(top_box.conf[0])
            class_id = int(top_box.cls[0])
            best_class = model.names[class_id]

        writer.writerow([frame_id, best_class, round(best_conf, 4), round(latency, 2)])
        
        annotated_frame = results[0].plot()
        out.write(annotated_frame)
        frame_id += 1

cap.release()
out.release()
print("Done")

Loading YOLOv8 model
Processing video
Done
